# AMMS 302 — Week 5: Relational Database & SQL (Part 1)
**Relational database · Database schema · Column types · PRIMARY KEY · SQLite · INSERT · SELECT · LIMIT**

> Self-study notebook — รันออฟไลน์บน Windows ด้วย `uv run jupyter lab` ได้ทันที (ดูวิธีติดตั้งแบบละเอียดใน [สไลด์สัปดาห์ที่ 5](./wk05.html) สไลด์ 7 & 12)

### 🎯 Learning objectives (CLO1/CLO2)
- อธิบายสคีมาและชนิดคอลัมน์ได้
- สร้างฐานข้อมูล SQLite ไฟล์เดียว (`healthinfo.db`) และกำหนด PRIMARY KEY/constraints ได้
- เขียน `CREATE TABLE` / `INSERT` / `SELECT` + `LIMIT` เพื่อสืบค้นเวชระเบียนได้

### 📚 Official references (เปิดคู่กับสไลด์)
- SQLite Docs: [sqlite.org/docs.html](https://www.sqlite.org/docs.html) · Datatypes: [datatype3.html](https://www.sqlite.org/datatype3.html) · CREATE TABLE: [lang_createtable.html](https://www.sqlite.org/lang_createtable.html) · INSERT: [lang_insert.html](https://www.sqlite.org/lang_insert.html) · SELECT: [lang_select.html](https://www.sqlite.org/lang_select.html) · CLI: [cli.html](https://www.sqlite.org/cli.html)
- Python: [sqlite3 — Python docs](https://docs.python.org/3/library/sqlite3.html) · [sqlite3 tutorial](https://docs.python.org/3/library/sqlite3.html#tutorial)
- Tools: [DB Browser for SQLite](https://sqlitebrowser.org/) (`scoop install sqlitebrowser`) — สไลด์ 7 & 12
- Supplementary: [W3Schools SQL Tutorial](https://www.w3schools.com/sql/) · MIMIC-IV hosp tables: [mimic.mit.edu/docs/iv/modules/hosp/](https://mimic.mit.edu/docs/iv/modules/hosp/) · OMOP CDM: [ohdsi.github.io/CommonDataModel](https://ohdsi.github.io/CommonDataModel/)

---

### 🗺️ แผนที่สไลด์ ↔ โน้ตบุ๊ก (ใช้เรียนข้ามกันได้)

| ipynb § | หัวข้อ | สไลด์ wk05 |
|---|---|---|
| §2 | CREATE TABLE patients | 07 |
| §3–4 | INSERT + executemany CSV | 08 |
| §5 | SELECT + alias + pandas | 10 |
| §6 | LIMIT / OFFSET | 11 |
| §7 | IS NULL vs = NULL | 09 |
| §8 | PK IntegrityError + OR IGNORE | 05 |

> เปิดคู่กับ [wk05.html](./wk05.html) — Cheat sheet อยู่สไลด์ 15


## 0) เตรียมเครื่อง (Windows) — สรุปสั้น
สไลด์สัปดาห์ที่ 5 สไลด์ 7 & 12 อธิบายแบบละเอียดแล้ว — สรุปคำสั่งที่ใช้ใน PowerShell:

```powershell
# ครั้งแรกเท่านั้น (ถ้ายังไม่มี scoop)
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
Invoke-RestMethod -Uri https://get.scoop.sh | Invoke-Expression
scoop install git uv sqlite sqlitebrowser

# ในโฟลเดอร์แล็บ (เช่น D:\health-informatics)
uv init          # ถ้ายังไม่มี pyproject.toml
uv add jupyterlab pandas
uv run jupyter lab   # เปิดโน้ตบุ๊กนี้
```
> SQLite มากับ Python อยู่แล้ว (`import sqlite3` ได้เลย) — `scoop install sqlite` เอาไว้ใช้ CLI `sqlite3 healthinfo.db` เท่านั้น


In [ ]:
# ตรวจสอบเวอร์ชัน — รันเซลล์นี้ก่อนเสมอ
import sqlite3, sys, pathlib
import pandas as pd
print(f"Python {sys.version.split()[0]}")
print(f"sqlite3 {sqlite3.sqlite_version}  (DB-API {sqlite3.version})")
print(f"pandas {pd.__version__}")
print(f"working dir: {pathlib.Path.cwd()}")
# ไฟล์ข้อมูลที่ต้องมีในโฟลเดอร์เดียวกับ notebook:
print("patients_data.csv exists:", pathlib.Path("patients_data.csv").exists())
print("patient_fhir_demo.json exists:", pathlib.Path("patient_fhir_demo.json").exists())

## 1) สร้างฐานข้อมูลไฟล์เดียว `healthinfo.db`
SQLite เก็บทั้งฐานในไฟล์เดียว — ลบไฟล์ = ล้างฐาน (สไลด์ 7: One-file DB — [sqlite.org/onefile.html](https://www.sqlite.org/onefile.html))

**Diagram:**
```
healthinfo.db  ──▶  [patients]  [prescriptions]  [visits]  (หลายตารางในไฟล์เดียว)
```
ถ้าไฟล์มีอยู่แล้วจากการรันก่อนหน้า เราจะลบแล้วสร้างใหม่เพื่อให้ผลซ้ำได้ (reproducible)


In [ ]:
import pathlib
db_path = pathlib.Path("healthinfo.db")
# ลบไฟล์เก่าเพื่อให้รันซ้ำได้เหมือนเดิม (ถ้าต้องการเก็บข้อมูลเดิม ให้คอมเมนต์บรรทัดล่าง)
if db_path.exists():
    db_path.unlink()
    print(f"removed old {db_path}")

con = sqlite3.connect(db_path)  # สร้างไฟล์ใหม่ถ้ายังไม่มี
cur = con.cursor()
print(f"connected to {db_path.resolve()}")
print(f"SQLite file size: {db_path.stat().st_size} bytes" if db_path.exists() else "(new file)")

## 2) `CREATE TABLE` — นิยามสคีมาตารางผู้ป่วย
อ้างอิงสเปก: [CREATE TABLE — lang_createtable.html](https://www.sqlite.org/lang_createtable.html) · [PRIMARY KEY](https://www.sqlite.org/lang_createtable.html#primkeyconst) · ชนิดข้อมูล: [datatype3.html](https://www.sqlite.org/datatype3.html)

Data Dictionary ของเร (ย่อจาก สปสช./MIMIC):

| คอลัมน์ | Type | Constraint | ความหมาย |
|---|---|---|---|
| patient_id | INTEGER | PRIMARY KEY | รหัสผู้ป่วย (ไม่ซ้ำ) |
| hn | TEXT | UNIQUE NOT NULL | เลข HN |
| gender | TEXT | CHECK IN ('ชาย','หญิง') | เพศ |
| birth_date | TEXT | — | วันเกิด ISO8601 YYYY-MM-DD |
| systolic_bp | REAL | CHECK >0 | ความดันบน mmHg |
| hba1c | REAL | — | HbA1c % (อาจเป็น NULL) |

ทำไม `birth_date` เป็น TEXT? SQLite ไม่มี DATE type แยก — เก็บเป็น ISO8601 TEXT แล้วใช้ [date/time functions](https://www.sqlite.org/lang_datefunc.html) ได้


In [ ]:
cur.execute("""
CREATE TABLE IF NOT EXISTS patients(
  patient_id   INTEGER PRIMARY KEY,          -- บัตรประชาชนของแถว (rowid alias, เร็วสุด)
  hn           TEXT UNIQUE NOT NULL,         -- ห้ามซ้ำ/ห้ามว่าง
  gender       TEXT CHECK(gender IN ('ชาย','หญิง')),
  birth_date   TEXT,                         -- ISO8601: 1960-06-23
  systolic_bp  REAL CHECK(systolic_bp > 0),
  hba1c        REAL                          -- NULL = ยังไม่ได้วัด
);
""")
con.commit()
print("CREATE TABLE done")

### ตรวจสอบสคีมที่สร้าง — เหมือนสไลด์ 4 (`.schema`)
ใน CLI จะพิมพ์ `.schema patients` — ใน Python ดูจาก `sqlite_master` ได้


In [ ]:
cur.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='patients'")
print(cur.fetchone()[0])

# ดูคอลัมน์แบบละเอียดด้วย PRAGMA (สไลด์ 6)
display(pd.read_sql("PRAGMA table_info(patients)", con))

## 3) `INSERT` — เพิ่มข้อมูลทีละแถว
สเปก: [INSERT — lang_insert.html](https://www.sqlite.org/lang_insert.html) · [executemany — Python docs](https://docs.python.org/3/library/sqlite3.html#sqlite3.Cursor.executemany)

⚠️ ต้อง `con.commit()` หลัง INSERT — ไม่งั้นข้อมูลไม่บันทึก (สไลด์ 9)


In [ ]:
# เพิ่ม 3 แถวด้วยมือ — เหมือนเขียนใบรับผู้ป่วยใหม่
cur.execute("""
INSERT INTO patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c)
VALUES (10001, 'HN-54001', 'ชาย', '1960-06-23', 129.0, 8.1)
""")
cur.execute("""
INSERT INTO patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c)
VALUES (10002, 'HN-54002', 'หญิง', '1975-02-10', 155.0, 8.3)
""")
cur.execute("""
INSERT INTO patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c)
VALUES (10003, 'HN-54003', 'หญิง', '1970-01-25', 140.0, NULL)  -- HbA1c ยังไม่ได้วัด
""")
con.commit()
print(f"inserted 3 rows, now count = {cur.execute('SELECT COUNT(*) FROM patients').fetchone()[0]}")

## 4) `INSERT` หลายแถวจาก CSV — โหลด `patients_data.csv` (100 แถว)
ไฟล์ `patients_data.csv` อยู่โฟลเดอร์เดียวกับ notebook (สไลด์ 8–9) — เราใช้ `csv.DictReader` + `executemany` เพื่อให้เร็วและปลอดภัย (parameterized query ป้องกัน SQL injection)

CSV ต้นฉบับมีคอลัมน์: `subject_id,Name,Age,gender,systolic_bp,HbA1c_level,admission_date` — เราจะ map เฉพาะที่ตรงกับสคีมา และสร้าง `hn` แบบ `HN-<patient_id>` เพื่อให้ UNIQUE


In [ ]:
import csv

inserted = 0
skipped = 0
with open("patients_data.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    # เตรียม rows ให้ตรงกับสคีมา patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c)
    # birth_date ใน CSV คือ admission_date — ใช้เป็น proxy; ถ้าเป็น NULL ให้เป็น None
    batch = []
    for r in reader:
        try:
            pid = int(float(r["subject_id"])) if r["subject_id"] else None
        except:
            continue
        if pid is None or pid in (10001, 10002, 10003):  # ข้าม 3 คนที่ใส่ไปแล้ว (PK ซ้ำ)
            skipped += 1
            continue
        hn = f"HN-{pid}"
        gender = r["gender"] if r["gender"] in ("ชาย", "หญิง") else None
        birth_date = r["admission_date"] if r["admission_date"] and "-" in r["admission_date"] else None  # ข้ามรูปแบบ 24/05/2025 ที่ผิด ISO
        try:
            sbp = float(r["systolic_bp"]) if r["systolic_bp"] else None
        except:
            sbp = None
        try:
            hba1c = float(r["HbA1c_level"]) if r["HbA1c_level"] else None
        except:
            hba1c = None
        batch.append((pid, hn, gender, birth_date, sbp, hba1c))

    # ใช้ OR IGNORE เพื่อข้ามแถวที่ละเมิด PK/UNIQUE โดยไม่ error ทั้ง batch (สไลด์ 9)
    cur.executemany("""
        INSERT OR IGNORE INTO patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c)
        VALUES (?, ?, ?, ?, ?, ?)
    """, batch)
    inserted = cur.rowcount if cur.rowcount != -1 else len(batch)
    con.commit()

count = cur.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
print(f"batch size: {len(batch)}, skipped PK duplicate: {skipped}")
print(f"total rows in patients: {count} (expected ~103 = 3 + 100, แต่ CSV มีแถวซ้ำ/PK ชนจึงน้อยกว่าเล็กน้อยได้)")

## 5) `SELECT` — สืบค้นพื้นฐาน (สไลด์ 10)
ไวยากรณ์: `SELECT คอลัมน์ FROM ตาราง;` — สเปก: [SELECT — lang_select.html](https://www.sqlite.org/lang_select.html) · [W3Schools SELECT](https://www.w3schools.com/sql/sql_select.asp)

ลำดับการทำงานจริง: `FROM` → `SELECT` (ไม่ใช่ SELECT ก่อน) — เขียน `SELECT *` เฉพาะตอนสำรวจ, งานจริงควรระบุคอลัมน์เพื่อประหยัด RAM


In [ ]:
# 5.1 ทุกคอลัมน์ ทุกแถว — อย่ารันกับตารางใหญ่โดยไม่มี LIMIT!
print("-- 5.1 SELECT * (ดู 3 แถวแรกผ่าน LIMIT เพื่อไม่ให้จอแตก) --")
for row in cur.execute("SELECT * FROM patients LIMIT 3"):
    print(row)

# 5.2 เลือกเฉพาะคอลัมน์ที่ต้องการ
print("\n-- 5.2 SELECT hn, hba1c --")
for row in cur.execute("SELECT hn, hba1c FROM patients LIMIT 3"):
    print(row)

# 5.3 ใช้ Alias ให้อ่านง่าย
print("\n-- 5.3 SELECT hn AS hospital_no --")
for row in cur.execute("SELECT hn AS hospital_no, hba1c FROM patients LIMIT 3"):
    print(row)

# 5.4 ผ่าน pandas — คุ้นเคยจากสัปดาห์ 1–3
print("\n-- 5.4 via pandas read_sql --")
display(pd.read_sql("SELECT hn, hba1c FROM patients LIMIT 5", con))

## 6) `LIMIT` / `OFFSET` — แบ่งหน้า & สุ่มตรวจ (สไลด์ 11)
ข้อมูลสุขภาพมีแสน–ล้านแถว — `LIMIT n` ดูตัวอย่าง, `LIMIT 10 OFFSET 20` แบ่งหน้า — สเปก: [SELECT → LIMIT](https://www.sqlite.org/lang_select.html#limitopts) · [W3Schools TOP/LIMIT](https://www.w3schools.com/sql/sql_top.asp)

> ⚠️ `LIMIT` โดยไม่มี `ORDER BY` ลำดับอาจสุ่ม — ถ้าต้องการซ้ำได้ให้ใส่ `ORDER BY patient_id`


In [ ]:
print("-- 6.1 LIMIT 5 -- 5 แถวแรก --")
display(pd.read_sql("SELECT * FROM patients ORDER BY patient_id LIMIT 5", con))

print("-- 6.2 LIMIT 5 OFFSET 5 -- หน้าที่ 2 (แถว 6-10) --")
display(pd.read_sql("SELECT patient_id, hn FROM patients ORDER BY patient_id LIMIT 5 OFFSET 5", con))

print("-- 6.3 เทียบกับ patients_data.csv ผ่าน pandas โดยตรง (ไม่ผ่าน SQL) — ผลควรตรงกัน --")
print(pd.read_csv("patients_data.csv", nrows=5).head())

## 7) `NULL` — ค่าว่างคนละแบบกับ `''` (สไลด์ 9)
ในเวชระเบียน `NULL` = ไม่ได้วัด/ไม่ทราบ — ตรวจด้วย `IS NULL` เท่านั้น (`= NULL` จะได้ 0 แถวเสมอ!)


In [ ]:
print("-- 7.1 หาผู้ป่วยที่ยังไม่ได้วัด HbA1c --")
display(pd.read_sql("SELECT patient_id, hn, hba1c FROM patients WHERE hba1c IS NULL LIMIT 5", con))
count_null = cur.execute("SELECT COUNT(*) FROM patients WHERE hba1c IS NULL").fetchone()[0]
count_not_null = cur.execute("SELECT COUNT(*) FROM patients WHERE hba1c IS NOT NULL").fetchone()[0]
print(f"NULL: {count_null}, NOT NULL: {count_not_null}, total: {count_null+count_not_null}")

print("\n-- 7.2 ผิด: WHERE hba1c = NULL  → ได้ 0 แถวเสมอ (สาธิต) --")
print(pd.read_sql("SELECT COUNT(*) AS cnt FROM patients WHERE hba1c = NULL", con))

## 8) ทดสอบ PRIMARY KEY — ความเป็นเอกลักษณ์ (สไลด์ 6)
PK ต้อง UNIQUE + NOT NULL — ถ้า INSERT ซ้ำจะได้ `UNIQUE constraint failed` ทันที (ป้องกันข้อมูลซ้ำ) — สเปก: [PRIMARY KEY](https://www.sqlite.org/lang_createtable.html#primkeyconst)


In [ ]:
try:
    cur.execute("INSERT INTO patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c) VALUES (10001, 'HN-99999', 'ชาย', '2000-01-01', 120, 5.5)")
    con.commit()
    print("unexpected: insert succeeded (should have failed)")
except sqlite3.IntegrityError as e:
    print(f"✅ IntegrityError (คาดไว้): {e}")
    print("→ PK ป้องกันการเพิ่ม patient_id ซ้ำได้ถูกต้อง")

try:
    cur.execute("INSERT INTO patients(patient_id, hn, gender, birth_date, systolic_bp, hba1c) VALUES (99999, 'HN-54001', 'หญิง', '2000-01-01', 120, 5.5)")
    con.commit()
except sqlite3.IntegrityError as e:
    print(f"✅ UNIQUE hn failed (คาดไว้): {e}")

# เทียบกับ INSERT OR IGNORE — จะไม่ error แต่ข้ามแถวซ้ำ
cur.execute("INSERT OR IGNORE INTO patients VALUES (10001, 'HN-54001', 'ชาย', '1960-06-23', 129, 8.1)")
print(f"OR IGNORE: rowcount={cur.rowcount}, count still {cur.execute('SELECT COUNT(*) FROM patients').fetchone()[0]}")

## 9) ปิดการเชื่อมต่อ & ส่งงาน
ไฟล์ `healthinfo.db` อยู่โฟลเดอร์เดียวกับ notebook — ก๊อปปี้ไฟล์นี้ส่งอาจารย์ได้เลย (สไลด์ 7: One-file DB)


In [ ]:
# ตรวจขนาดไฟล์ก่อนปิด
import pathlib
db_path = pathlib.Path("healthinfo.db")
print(f"DB: {db_path.resolve()} — {db_path.stat().st_size/1024:.1f} KB — tables: {cur.execute(\"SELECT name FROM sqlite_master WHERE type='table'\").fetchall()}")
con.commit()
con.close()
print("connection closed — เปิดใหม่ด้วย sqlite3.connect('healthinfo.db') ได้")

# ทดสอบเปิดใหม่ด้วย pandas (เหมือนสไลด์ 10)
con2 = sqlite3.connect("healthinfo.db")
print(pd.read_sql("SELECT COUNT(*) AS total_patients FROM patients", con2))
con2.close()

### 🧾 Cheat Sheet — Week 5 (ตรงกับสไลด์ 15)

| งาน | SQL / Python |
|---|---|
| สร้างตาราง | `CREATE TABLE IF NOT EXISTS t(col TYPE RULE,…)` |
| เพิ่มแถว | `INSERT [OR IGNORE] INTO t VALUES(?)` |
| หลายแถว | `cur.executemany(sql, rows)` |
| อ่าน | `SELECT cols FROM t [LIMIT n]` |
| บันทึก | `con.commit()` — ลืม = ข้อมูลหาย! |
| ดูสคีมา | `.schema t` / `PRAGMA table_info(t)` |
| นับแถว | `SELECT COUNT(*) FROM t` |

**Error ที่เจอบ่อย:** `UNIQUE constraint failed` → id ซ้ำ (ใช้ OR IGNORE) · `no such column/table` → พิมพ์ผิด/ลืม CREATE · WHERE x = NULL ได้ 0 แถว → ใช้ `IS NULL`

**Pipeline Python:** `connect → cursor → execute/executemany → commit → close`


### ✅ Self-check (เกณฑ์ตรวจ — สไลด์ 13)
- `SELECT COUNT(*) FROM patients;` → ~103 แถว (3 แถวมือ + ~100 จาก CSV, บางแถวถูก `OR IGNORE` ข้ามเพราะ PK ซ้ำ)
- `SELECT sql FROM sqlite_master;` เห็น `PRIMARY KEY` / `UNIQUE` / `CHECK` ครบ
- `SELECT * FROM patients LIMIT 5;` ผลตรงกับ `pd.read_sql(..., con)` และ `pd.read_csv("patients_data.csv").head()`
- ลอง `INSERT` PK ซ้ำแล้วได้ `UNIQUE constraint failed` และอธิบายได้

### 📝 Homework 5 (ถ้าอาจารย์สั่ง)
ส่ง `week05-sql-basics.ipynb` + `healthinfo.db` — อาจารย์ตรวจด้วย:
```bash
sqlite3 healthinfo.db "SELECT COUNT(*) FROM patients;"
sqlite3 healthinfo.db ".schema patients"
```

### 🔜 สัปดาห์หน้า: Week 6 — WHERE, ORDER BY, Aggregation (`MIN/MAX/AVG/COUNT`)
เตรียม `healthinfo.db` ไว้ — เราจะสืบค้นเชิงลึกด้วย `WHERE` + `ORDER BY` ต่อ

---
### 🔗 รวมลิงก์สเปกทางการ (ซ้ำจากสไลด์ 14 — เก็บไว้ใน notebook เพื่อ self-study)
- [SQLite Documentation](https://www.sqlite.org/docs.html) · [Datatypes](https://www.sqlite.org/datatype3.html) · [CREATE TABLE](https://www.sqlite.org/lang_createtable.html) · [INSERT](https://www.sqlite.org/lang_insert.html) · [SELECT](https://www.sqlite.org/lang_select.html) · [CLI](https://www.sqlite.org/cli.html) · [One-file DB](https://www.sqlite.org/onefile.html)
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) · [sqlite3 tutorial](https://docs.python.org/3/library/sqlite3.html#tutorial)
- [DB Browser for SQLite](https://sqlitebrowser.org/) · [W3Schools SQL Tutorial](https://www.w3schools.com/sql/)
- สไลด์: [wk05.html](./wk05.html) — สไลด์ 5 (Types), 6 (PK), 8 (CREATE), 9 (INSERT), 10–11 (SELECT/LIMIT), 12 (Workflow)
